In [18]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import requests
from scipy.optimize import fsolve
from scipy.interpolate import CubicSpline

def fetch_euribor_ois_data():
    """
    Fetch data from FRED and other free sources.
    Note: Full OIS curve isn't freely available, so we'll use proxies.
    """
    
    # For demonstration, I'll create realistic data
    # In production, you'd use FRED API: https://fred.stlouisfed.org/docs/api/fred/
    
    # Settlement date
    today = datetime(2026, 6, 13)  # Example date
    
    # OIS quotes (€STR) - These come from professional terminals
    # You would fetch these from ECB's MIR or professional vendors
    ois_maturities = [7, 30, 90, 180, 365, 730, 1095, 1825, 3650]  # days
    ois_rates = [2.85, 2.88, 2.92, 2.95, 3.00, 3.05, 3.10, 3.20, 3.35]  # percentages
    
    # EURIBOR 3M swap quotes (for projection curve)
    swap_maturities = [365, 730, 1095, 1825, 2738, 3650, 5475, 7300]  # days
    swap_rates = [3.10, 3.25, 3.40, 3.65, 3.85, 4.00, 4.20, 4.35]  # percentages
    
    return {
        'today': today,
        'ois_maturities': np.array(ois_maturities),
        'ois_rates': np.array(ois_rates) / 100.0,
        'swap_maturities': np.array(swap_maturities),
        'swap_rates': np.array(swap_rates) / 100.0
    }

# Alternative: Actually fetch from FRED using their API (if you have a key)
def fetch_from_fred(api_key=None):
    """
    Example of fetching EURIBOR from FRED
    FRED series: 'EURIBOR3M' for 3-month Euribor
    """
    if api_key:
        url = f"https://api.stlouisfed.org/fred/series/observations?series_id=EURIBOR3M&api_key={api_key}&file_type=json"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return pd.DataFrame(data['observations'])
    return None

In [19]:
class DayCounter:
    """Handle different day count conventions"""
    
    @staticmethod
    def actual_360(start_date, end_date):
        """Actual/360: Used for EURIBOR"""
        days = (end_date - start_date).days
        return days / 360.0
    
    @staticmethod
    def actual_365_fixed(start_date, end_date):
        """Actual/365 Fixed: Used for OIS discounting"""
        days = (end_date - start_date).days
        return days / 365.0
    
    @staticmethod
    def thirty_360(start_date, end_date):
        """30/360: Used for fixed leg of EUR swaps"""
        # Simplified 30/360
        d1, m1, y1 = start_date.day, start_date.month, start_date.year
        d2, m2, y2 = end_date.day, end_date.month, end_date.year
        
        d1 = min(d1, 30)
        d2 = min(d2, 30) if d2 == 31 else d2
        
        days = (y2 - y1) * 360 + (m2 - m1) * 30 + (d2 - d1)
        return days / 360.0

def generate_payment_dates(start_date, maturity_days, frequency_months=12):
    """Generate payment dates for a swap"""
    dates = [start_date]
    current = start_date
    
    while (current - start_date).days < maturity_days:
        # Add frequency_months months
        year = current.year
        month = current.month + frequency_months
        day = current.day
        
        while month > 12:
            month -= 12
            year += 1
        
        # Handle month-end issues (simplified)
        day = min(day, [31, 28 + (year % 4 == 0), 31, 30, 31, 30, 
                        31, 31, 30, 31, 30, 31][month - 1])
        
        current = datetime(year, month, day)
        if (current - start_date).days <= maturity_days:
            dates.append(current)
    
    return dates

In [20]:
class OISCurveBootstrap:
    """Bootstrap discount curve from €STR OIS swaps"""
    
    def __init__(self, today, maturities, ois_quotes):
        self.today = today
        self.maturities = maturities
        self.quotes = ois_quotes
        self.discount_factors = np.ones(len(maturities) + 1)
        self.times = np.zeros(len(maturities) + 1)
        
    def bootstrap(self):
        """
        Bootstrap OIS curve.
        OIS swaps pay fixed vs floating overnight rate.
        For OIS: PV(fixed leg) = PV(floating leg)
        """
        n = len(self.maturities)
        self.times[0] = 0
        self.discount_factors[0] = 1.0
        
        for i in range(n):
            # Calculate fixed leg PV
            fixed_pv = self._calculate_fixed_leg_pv(i)
            
            # Floating leg PV = 1 - DF(maturity)
            # For OIS, floating leg PV = 1 - DF_maturity
            # This gives us the equation: fixed_pv = 1 - DF_maturity
            df_maturity = 1.0 - fixed_pv
            
            self.discount_factors[i + 1] = df_maturity
            self.times[i + 1] = self.maturities[i] / 365.0
            
            # Use interpolation for intermediate points if needed
            if i > 0:
                self._interpolate_missing_points(i)
        
        return self.times, self.discount_factors
    
    def _calculate_fixed_leg_pv(self, idx):
        """Calculate PV of fixed leg for OIS"""
        maturity_days = self.maturities[idx]
        fixed_rate = self.quotes[idx]
        
        # Generate payment dates (typically annual for OIS)
        payment_dates = generate_payment_dates(self.today, maturity_days, 12)
        
        fixed_pv = 0.0
        for i in range(1, len(payment_dates)):
            start = payment_dates[i-1]
            end = payment_dates[i]
            
            # Day count for fixed leg (30/360 or Actual/365)
            tau = DayCounter.actual_365_fixed(start, end)
            
            # Discount factor at payment date (interpolated from known points)
            df = self._interpolate_df((end - self.today).days)
            
            fixed_pv += fixed_rate * tau * df
        
        return fixed_pv
    
    def _interpolate_df(self, days):
        """Interpolate discount factor from bootstrapped points"""
        # Use log-linear interpolation on discount factors
        time = days / 365.0
        
        if time <= self.times[0]:
            return 1.0
        
        # Find surrounding points
        idx = np.searchsorted(self.times[self.times > 0], time)
        if idx >= len(self.times) or self.times[idx] == 0:
            idx = len(self.times) - 1
            while idx >= 0 and self.times[idx] == 0:
                idx -= 1
        
        if idx <= 0:
            return self.discount_factors[0]
        
        t1, t2 = self.times[idx-1], self.times[idx]
        df1, df2 = self.discount_factors[idx-1], self.discount_factors[idx]
        
        # Log-linear interpolation
        if t2 > t1:
            log_df = np.log(df1) + (np.log(df2) - np.log(df1)) * (time - t1) / (t2 - t1)
            return np.exp(log_df)
        else:
            return df1
    
    def _interpolate_missing_points(self, up_to_idx):
        """Create smooth interpolation between points"""
        # For simplicity, we keep only bootstrapped points
        # In practice, you'd use cubic spline interpolation
        pass

    def get_zero_rates(self):
        """Convert discount factors to zero rates (continuous compounding)"""
        zero_rates = np.zeros(len(self.times))
        for i in range(1, len(self.times)):
            if self.times[i] > 0:
                zero_rates[i] = -np.log(self.discount_factors[i]) / self.times[i]
        return zero_rates

In [21]:
class EuriborCurveBootstrap:
    """Bootstrap Euribor forward curve using discount curve"""
    
    def __init__(self, today, maturities, swap_quotes, discount_curve):
        self.today = today
        self.maturities = maturities
        self.quotes = swap_quotes
        self.discount_curve = discount_curve  # OIS curve from above
        self.discount_factors = None  # Will store DF at each payment date
        
        # Results
        self.forward_rates = np.ones(len(maturities))  # Forward rates between pillars
        self.zero_rates = np.ones(len(maturities))
        
    def bootstrap(self):
        """
        Bootstrap Euribor forward curve.
        Uses the relationship: fixed_leg_pv = float_leg_pv
        Float leg PV = sum(DF * tau * forward_rate)
        """
        n = len(self.maturities)
        cumulative_float_pv = 0.0
        
        for i in range(n):
            # For each new swap, solve for the forward rate that makes PV=0
            # The forward rate applies to the last period
            
            # Get payment schedule
            payment_dates = generate_payment_dates(self.today, self.maturities[i], 3)  # Quarterly for Euribor 3M
            
            # Get discount factors for all payment dates
            dfs = []
            times = []
            for date in payment_dates[1:]:  # Skip start date
                days = (date - self.today).days
                df = self._get_discount_factor(days)
                dfs.append(df)
                times.append(days / 365.0)
            
            # Fixed leg PV
            fixed_rate = self.quotes[i]
            fixed_pv = 0.0
            for j, df in enumerate(dfs):
                # Fixed leg payment dates (typically annual)
                tau_fixed = DayCounter.thirty_360(payment_dates[j], payment_dates[j+1]) if j+1 < len(payment_dates) else 0.25
                fixed_pv += fixed_rate * tau_fixed * df
            
            # Float leg PV = Sum of discounted forward rates
            # We've already computed forward rates for previous periods
            float_pv_known = cumulative_float_pv
            
            # The unknown forward rate for the last period
            # Solve: fixed_pv = float_pv_known + forward_rate * tau_float * last_df
            last_period_tau = DayCounter.actual_360(payment_dates[-2], payment_dates[-1])
            last_df = dfs[-1]
            
            forward_rate_new = (fixed_pv - float_pv_known) / (last_period_tau * last_df)
            
            self.forward_rates[i] = max(forward_rate_new, 0.0)  # No negative rates
            
            # Update cumulative float PV
            cumulative_float_pv += forward_rate_new * last_period_tau * last_df
            
            # Compute zero rate for this maturity
            total_time = self.maturities[i] / 365.0
            self.zero_rates[i] = -np.log(self._get_discount_factor(self.maturities[i])) / total_time
        
        return self.forward_rates, self.zero_rates
    
    def _get_discount_factor(self, days):
        """Get discount factor from OIS curve"""
        return self.discount_curve._interpolate_df(days)

In [22]:
def build_multi_curve_pipeline():
    """Complete pipeline to build both curves"""
    
    # 1. Fetch data
    data = fetch_euribor_ois_data()
    
    print("=" * 60)
    print("MULTI-CURVE BOOTSTRAP FOR EUR IRS")
    print("=" * 60)
    print(f"\nValuation Date: {data['today'].strftime('%Y-%m-%d')}")
    
    # 2. Build OIS discount curve
    print("\n1. Building €STR OIS Discount Curve...")
    ois_bootstrap = OISCurveBootstrap(
        data['today'], 
        data['ois_maturities'], 
        data['ois_rates']
    )
    times, dfs = ois_bootstrap.bootstrap()
    ois_zero_rates = ois_bootstrap.get_zero_rates()
    
    # Display OIS curve
    print("\n   €STR OIS Curve Results:")
    print(f"   {'Maturity':<12} {'Time (Y)':<10} {'Discount Factor':<18} {'Zero Rate (%)':<12}")
    print(f"   {'-'*50}")
    for i in range(len(times)):
        if i == 0:
            print(f"   {'Spot':<12} {times[i]:<10.4f} {dfs[i]:<18.6f} {'N/A':<12}")
        else:
            print(f"   {data['ois_maturities'][i-1]:<12} {times[i]:<10.4f} {dfs[i]:<18.6f} {ois_zero_rates[i]*100:<12.4f}")
    
    # 3. Build Euribor forward curve using OIS discounting
    print("\n2. Building EURIBOR 3M Forward Curve...")
    euribor_bootstrap = EuriborCurveBootstrap(
        data['today'],
        data['swap_maturities'],
        data['swap_rates'],
        ois_bootstrap
    )
    forward_rates, euribor_zero_rates = euribor_bootstrap.bootstrap()
    
    # Display forward curve
    print("\n   EURIBOR 3M Forward Curve Results:")
    print(f"   {'Maturity (Days)':<16} {'Time (Y)':<10} {'3M Forward Rate (%)':<20} {'Zero Rate (%)':<12}")
    print(f"   {'-'*60}")
    for i in range(len(forward_rates)):
        time_years = data['swap_maturities'][i] / 365.0
        print(f"   {data['swap_maturities'][i]:<16} {time_years:<10.2f} {forward_rates[i]*100:<20.4f} {euribor_zero_rates[i]*100:<12.4f}")
    
    # 4. Example: Price an IRS using the curves
    print("\n3. Example: Pricing a 5-year EURIBOR Receiver Swap")
    notional = 10_000_000  # €10M
    fixed_rate = 3.50  # 3.5% fixed receive
    maturity = 1825  # 5 years
    
    # Generate payment schedule
    payment_dates = generate_payment_dates(data['today'], maturity, 3)  # Quarterly
    
    # Calculate swap value
    fixed_leg_pv = 0
    float_leg_pv = 0
    
    for i in range(1, len(payment_dates)):
        start, end = payment_dates[i-1], payment_dates[i]
        
        # Day counts
        tau_float = DayCounter.actual_360(start, end)
        tau_fixed = DayCounter.thirty_360(start, end)
        
        # Discount factor from OIS curve
        days_to_end = (end - data['today']).days
        df = ois_bootstrap._interpolate_df(days_to_end)
        
        # Forward rate from Euribor curve (simplified - would need interpolation)
        # For demonstration, use the rate for the appropriate tenor
        time_to_start = (start - data['today']).days / 365.0
        time_to_end = days_to_end / 365.0
        
        # Find closest forward rate from bootstrapped points
        idx = np.searchsorted(data['swap_maturities'] / 365.0, time_to_end)
        if idx >= len(forward_rates):
            idx = len(forward_rates) - 1
        
        forward_rate = forward_rates[idx]
        
        # PV calculations
        fixed_leg_pv += fixed_rate / 100.0 * tau_fixed * df
        float_leg_pv += forward_rate * tau_float * df
    
    swap_value = notional * (fixed_leg_pv - float_leg_pv)  # Receiver: receive fixed, pay float
    
    print(f"   Notional: €{notional:,.0f}")
    print(f"   Fixed Rate Received: {fixed_rate}%")
    print(f"   Fixed Leg PV: {fixed_leg_pv:.6f}")
    print(f"   Float Leg PV: {float_leg_pv:.6f}")
    print(f"   Swap NPV: €{swap_value:,.2f}")
    
    # 5. Show the spread (critical for your earlier question)
    print("\n4. EURIBOR - OIS Spread Analysis:")
    print(f"   {'Maturity':<12} {'OIS Zero Rate':<18} {'EURIBOR Zero Rate':<20} {'Spread (bp)':<12}")
    print(f"   {'-'*60}")
    
    for i in range(min(len(ois_zero_rates), len(euribor_zero_rates))):
        if i > 0:  # Skip spot
            ois_rate = ois_zero_rates[i] * 100
            eur_rate = euribor_zero_rates[i] * 100
            spread = (eur_rate - ois_rate) * 100  # In basis points
            print(f"   {data['ois_maturities'][i-1]:<12} {ois_rate:<18.4f} {eur_rate:<20.4f} {spread:<12.2f}")
    
    return ois_bootstrap, euribor_bootstrap

# Run the pipeline
if __name__ == "__main__":
    ois_curve, euribor_curve = build_multi_curve_pipeline()

MULTI-CURVE BOOTSTRAP FOR EUR IRS

Valuation Date: 2026-06-13

1. Building €STR OIS Discount Curve...

   €STR OIS Curve Results:
   Maturity     Time (Y)   Discount Factor    Zero Rate (%)
   --------------------------------------------------
   Spot         0.0000     1.000000           N/A         
   7            0.0192     1.000000           -0.0000     
   30           0.0822     1.000000           -0.0000     
   90           0.2466     1.000000           -0.0000     
   180          0.4932     1.000000           -0.0000     
   365          1.0000     0.970000           3.0459      
   730          2.0000     0.969500           1.5487      
   1095         3.0000     0.938863           2.1028      
   1825         5.0000     0.877758           2.6077      
   3650         10.0000    0.734362           3.0875      

2. Building EURIBOR 3M Forward Curve...

   EURIBOR 3M Forward Curve Results:
   Maturity (Days)  Time (Y)   3M Forward Rate (%)  Zero Rate (%)
   ------------------